# Day 43 · The neural bigram: the same answer, learned

**First-principles question:** Can gradient descent learn what counting gave us?

The full task (learn → build → record) is in this folder's `README.md`. This notebook is your workbench:
run cells top to bottom with **Shift + Enter**, fill in every cell marked **TODO**, and finish with the
**✅ self-check** cell at the end.

In [ ]:
import sys
from pathlib import Path

def find_root():
    """The fde-first-principles folder (it has progress.json)."""
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "progress.json").exists():
            return p
    raise RuntimeError("Open this notebook from inside your fde-first-principles folder.")

ROOT = find_root()
sys.path.insert(0, str(ROOT))  # so `import lab.…` works

import urllib.request
import numpy as np
import matplotlib.pyplot as plt

def download(url, name):
    """Download once into data/downloads/ (ignored by git), then reuse."""
    path = ROOT / "data" / "downloads" / name
    if not path.exists():
        path.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(url, path)
    return path.read_text(encoding="utf-8")

words = download("https://raw.githubusercontent.com/karpathy/makemore/master/names.txt", "names.txt").split()
chars = sorted(set("".join(words)))
stoi = {ch: i + 1 for i, ch in enumerate(chars)}; stoi["."] = 0      # "." marks the start and the end of a name
itos = {i: ch for ch, i in stoi.items()}
print(len(words), "names, e.g.", words[:5])
N = np.zeros((27, 27), dtype=np.int64)
for w in words:
    seq = ["."] + list(w) + ["."]
    for ch1, ch2 in zip(seq, seq[1:]):
        N[stoi[ch1], stoi[ch2]] += 1
import torch
import torch.nn.functional as F
counting_loss = float(-np.log((N / N.sum(1, keepdims=True))[N > 0]).dot(N[N > 0]) / N.sum())
print(f"counting model loss: {counting_loss:.4f}")

## 1 · Training pairs
Input = the current letter, target = the next letter.

In [ ]:
xs, ys = [], []
for w in words:
    seq = ["."] + list(w) + ["."]
    for ch1, ch2 in zip(seq, seq[1:]):
        xs.append(stoi[ch1]); ys.append(stoi[ch2])
xs, ys = torch.tensor(xs), torch.tensor(ys)
print(len(xs), "training pairs")

## 2 · One layer, trained by gradient descent

In [ ]:
g = torch.Generator().manual_seed(2147483647)
W = torch.randn((27, 27), generator=g, requires_grad=True)
x_onehot = F.one_hot(xs, num_classes=27).float()
losses = []

for step in range(200):
    # TODO forward:  logits = x_onehot @ W; probs = softmax over each row (logits.exp(), divide by row sums)
    #                loss = -(probs[torch.arange(len(ys)), ys]).log().mean() + 0.01 * (W ** 2).mean()
    # TODO backward: W.grad = None; loss.backward()
    # TODO update:   W.data += -50 * W.grad
    # TODO record:   losses.append(loss.item())
    pass

print(f"neural loss {losses[-1]:.4f} vs. counting loss {counting_loss:.4f}")

## 3 · Sample from the network

In [ ]:
g = torch.Generator().manual_seed(2147483647)
for _ in range(10):
    out, ix = [], 0
    while True:
        p = F.softmax(W[ix], dim=0)
        ix = torch.multinomial(p, num_samples=1, generator=g).item()
        if ix == 0:
            break
        out.append(itos[ix])
    print("".join(out))

**Why do counting and the network give (almost) the same answer?** …

In [ ]:
# ✅ Self-check: run me last. A clear message tells you what to fix.
assert len(losses) == 200 and losses[-1] < losses[0]
assert losses[-1] < counting_loss + 0.06, "The network should get close to the counting loss"
print("All checks passed ✅  Now write 'In my words' in README.md and record your video.")